# Quebec French Dialect Drift — Colab runner

Runs the [`quebec-french-drift`](https://github.com/) harness on a Colab GPU.

Use this when the bottleneck is your machine. On a free-tier T4 an 8B model does a cell in
2–4 s against ~50 s on a swap-bound laptop, and disk is ~80 GB against 4 GB.

**Run the cells in order.** Cell 2 (Drive) is the one people skip and regret: Colab sessions
die after ~90 minutes idle, and results written to Drive make that a non-event because the
harness resumes from whatever cells are already recorded.

## 1 · Check the GPU and install

If `nvidia-smi` reports no GPU, set **Runtime → Change runtime type → T4 GPU** first.

In [ ]:
!nvidia-smi || echo 'NO GPU — set Runtime > Change runtime type > T4 GPU'
!pip -q install 'transformers>=4.44' 'peft>=0.12' accelerate bitsandbytes sentencepiece
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2 · Persist results to Drive

The harness writes every cell to `results/raw/` as it completes and skips recorded cells on
the next run. Pointing that at Drive means a disconnected session costs you one cell, not
the whole run.

Skip this only for a throwaway experiment.

In [ ]:
USE_DRIVE = True  #@param {type:'boolean'}

import os, pathlib
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS = pathlib.Path('/content/drive/MyDrive/qfdrift-results')
    RESULTS.mkdir(parents=True, exist_ok=True)
    print('results ->', RESULTS)
else:
    RESULTS = None
    print('results stay on the ephemeral Colab disk')

## 3 · Get the harness

In [ ]:
REPO_URL = ''  #@param {type:'string'}

import pathlib, subprocess, os
if REPO_URL:
    if not pathlib.Path('/content/french-ai').exists():
        subprocess.run(['git','clone','--depth','1',REPO_URL,'/content/french-ai'], check=True)
    os.chdir('/content/french-ai')
else:
    # No repo URL: upload the project as a zip instead.
    from google.colab import files
    if not pathlib.Path('/content/french-ai').exists():
        up = files.upload()
        name = next(iter(up))
        subprocess.run(['unzip','-q',name,'-d','/content/french-ai'], check=True)
    os.chdir('/content/french-ai')

# Point results at Drive so a dropped session loses at most one cell.
if RESULTS is not None:
    local = pathlib.Path('results')
    if local.is_symlink() or local.exists():
        if not local.is_symlink():
            subprocess.run(['cp','-rn','results/.', str(RESULTS)], check=False)
            subprocess.run(['rm','-rf','results'], check=True)
    if not pathlib.Path('results').exists():
        os.symlink(RESULTS, 'results')

!python3 tests/test_harness.py

## 4 · Choose models

Any HuggingFace causal LM by repo id. Two things worth knowing before you pick:

**Instruction-tuned only.** The four prompt conditions are instructions. A *base* model
(no `-Instruct`/`-Chat` suffix, no chat template) cannot follow them, scores 100% void
cells, and tells you nothing about French — that is exactly what happened to the QuebecLLM
Llama adapters.

**Quantization is a variable.** A 4-bit run is not comparable with an fp16 run of the same
model. Keep it constant across anything you intend to compare.

In [ ]:
MODELS = [
    'mistralai/Mistral-7B-Instruct-v0.3',
    'Qwen/Qwen2.5-7B-Instruct',
    'meta-llama/Llama-3.1-8B-Instruct',   # gated: needs HF_TOKEN + accepted licence
]
LOAD_IN_4BIT = True   #@param {type:'boolean'}  # required for 8B on a 16 GB T4
CONDITIONS   = ''     #@param {type:'string'}   # e.g. 'baseline proofread'; blank = all four

# A gated repo needs a token AND the licence accepted on your account.
HF_TOKEN = ''  #@param {type:'string'}
import os
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

print(f'{len(MODELS)} models x 88 items x {len(CONDITIONS.split()) or 4} conditions'
      f' = {len(MODELS)*88*(len(CONDITIONS.split()) or 4)} cells')

## 5 · Run

Model-outer and sequential: each model is loaded once, drained, then freed. Interleaving
them would thrash GPU memory for nothing.

Safe to re-run after a disconnect — recorded cells are skipped.

In [ ]:
import subprocess, shlex

for repo in MODELS:
    cmd = ['python3','run_experiment.py','run','--models',f'transformers:{repo}','--device','cuda']
    if LOAD_IN_4BIT: cmd.append('--load-in-4bit')
    if CONDITIONS:   cmd += ['--conditions', *CONDITIONS.split()]
    print('\n' + ' '.join(shlex.quote(c) for c in cmd), flush=True)
    subprocess.run(cmd)

## 6 · Merge a LoRA adapter (optional)

For evaluating a dialect adapter. **Always run its unmerged base too** — without that
control you cannot tell an adapter's effect from a base model that never worked. That
distinction is the whole difference between the QuebecLLM CroissantLLM result (conclusive)
and the Llama ones (uninformative).

In [ ]:
ADAPTER = 'QuebecLLM/QC-CroissantLLM_6e_CPT'  #@param {type:'string'}
BASE    = ''  #@param {type:'string'}  # blank = whatever the adapter names

import subprocess
cmd = ['python3','scripts/merge_lora.py',ADAPTER,'/content/merged']
if BASE: cmd += ['--base', BASE]
subprocess.run(cmd, check=True)

# Merged model, then its base as the control.
subprocess.run(['python3','run_experiment.py','run','--models','transformers:/content/merged',
                '--device','cuda','--conditions','baseline'])
import json, pathlib
named = json.loads(pathlib.Path('/content/merged/MERGE_INFO.json').read_text())['base_used']
subprocess.run(['python3','run_experiment.py','run','--models',f'transformers:{named}',
                '--device','cuda','--conditions','baseline'])

## 7 · A commercial API (optional)

This addresses the study's biggest gap — every result so far is a local open-weight model.
Any OpenAI-compatible endpoint works, and it costs a few dollars rather than a GPU.

In [ ]:
API_BASE = 'https://api.openai.com/v1'  #@param {type:'string'}
API_KEY  = ''  #@param {type:'string'}
API_MODEL= 'gpt-4o-mini'  #@param {type:'string'}

if API_KEY:
    import subprocess
    subprocess.run(['python3','run_experiment.py','run','--models',f'openai:{API_MODEL}',
                    '--base-url',API_BASE,'--api-key',API_KEY])
else:
    print('set API_KEY to run this cell')

## 8 · Report

Metrics are recomputed from the stored outputs, so this reflects everything recorded —
including runs from earlier sessions if you used Drive.

Read the **void cells** column first. A model that never performs the rewrite substitutes
nothing and would score 0% drift, which reads as perfect preservation; the report withholds
that number above 50% void rather than printing it.

In [ ]:
!python3 run_experiment.py report

from IPython.display import Markdown, display
import pathlib
display(Markdown(pathlib.Path('results/report/report.md').read_text()))

## 9 · Blind CSV for human raters

Shuffled, with model and condition withheld and kept in a separate key file. Telling a
rater which condition they are seeing measures their expectations rather than the language.

In [ ]:
!python3 run_experiment.py human --sample 150
from google.colab import files
files.download('results/human/human_eval.csv')